# Boston Housing — Análisis Exploratorio Inicial (EDA)
**Pregunta general:** ¿Qué debemos conocer de un conjunto de datos antes de intentar modelarlo?

---

**Variables:** `CRIM`, `ZN`, `INDUS`, `CHAS`, `NOX`, `RM`, `AGE`, `DIS`, `RAD`, `TAX`, `PTRATIO`, `B`, `LSTAT` y `MEDV`.
`MEDV` será considerada la variable objetivo.

## 1. Preparación y carga de los datos

**Pregunta orientadora:** ¿Qué datos tenemos y cómo están organizados?

### 1.1 Importación

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### 1.2 Archivo de datos

Se asume que `boston.csv` se encuentra en el mismo directorio del notebook y que la primera fila contiene los nombres de las columnas.

In [ ]:
# Ruta del conjunto de datos
dataset_name = './boston.csv'

# Cargar los valores numéricos
datos = np.loadtxt(dataset_name, delimiter=',', skiprows=1)

# Leer los nombres de las columnas
with open(dataset_name, 'r', encoding='utf-8') as f:
    nombres_columnas = f.readline().strip().replace('"', '').split(',')

print('Columnas:', nombres_columnas)

### 1.3 Primera inspección

In [ ]:
print('Tamaño:', datos.shape)
print('Dimensiones:', datos.ndim)
print('Tipo de dato:', datos.dtype)
print('\nPrimeras 3 filas:')
print(datos[:3])

**Observación.** En la versión estándar el conjunto tiene **506 observaciones y 14 columnas**. Cada fila reúne las características de una observación y `MEDV` ocupa la última columna.

## 2. Calidad e integridad de los datos

**Pregunta orientadora:** ¿Podemos confiar en los datos antes de analizarlos?

### 2.1 Valores faltantes y no finitos

In [ ]:
# Conteo por columna
n_nan = np.isnan(datos).sum(axis=0)
n_inf = np.isinf(datos).sum(axis=0)

for nombre, nan, inf in zip(nombres_columnas, n_nan, n_inf):
    print(f'{nombre:8s}  NaN: {nan:3d}   Inf: {inf:3d}')

### 2.2 Filas duplicadas

In [ ]:
# np.unique permite identificar filas repetidas
_, conteos = np.unique(datos, axis=0, return_counts=True)
n_duplicadas = np.sum(conteos - 1)

print('Filas duplicadas:', n_duplicadas)

### 2.3 Número de valores distintos

La cantidad de valores distintos ayuda a reconocer variables binarias o discretas aunque estén almacenadas como números.

In [ ]:
for i, nombre in enumerate(nombres_columnas):
    print(f'{nombre:8s}: {np.unique(datos[:, i]).size:3d} valores distintos')

**Observaciones.**

- En el conjunto estándar no aparecen valores `NaN`, infinitos ni filas duplicadas.
- `CHAS` tiene solo dos valores (`0` y `1`), por lo que es una variable **binaria**.
- `RAD` contiene pocos valores posibles y funciona como un **índice discreto**.
- Que una variable esté almacenada como `float` no significa que deba interpretarse como continua.

## 3. Estadística descriptiva

**Pregunta orientadora:** ¿Cómo se comporta cada variable por separado?

### 3.1 Resumen general

Se calculan mínimo, primer cuartil, mediana, media, tercer cuartil, máximo y desviación estándar.

In [ ]:
minimos = np.min(datos, axis=0)
q1 = np.percentile(datos, 25, axis=0)
medianas = np.median(datos, axis=0)
medias = np.mean(datos, axis=0)
q3 = np.percentile(datos, 75, axis=0)
maximos = np.max(datos, axis=0)
desv = np.std(datos, axis=0)

print(f'{"Var":8s} {"Min":>9s} {"Q1":>9s} {"Mediana":>9s} {"Media":>9s} {"Q3":>9s} {"Max":>9s} {"Std":>9s}')
print('-' * 82)
for i, nombre in enumerate(nombres_columnas):
    print(f'{nombre:8s} {minimos[i]:9.3f} {q1[i]:9.3f} {medianas[i]:9.3f} '
          f'{medias[i]:9.3f} {q3[i]:9.3f} {maximos[i]:9.3f} {desv[i]:9.3f}')

### 3.2 Media frente a mediana

Una diferencia grande entre ambas puede advertir **asimetría** o presencia de valores extremos.

In [ ]:
# Diferencia relativa entre media y mediana
for i, nombre in enumerate(nombres_columnas):
    escala = np.abs(medianas[i]) if medianas[i] != 0 else 1.0
    diferencia = np.abs(medias[i] - medianas[i]) / escala
    print(f'{nombre:8s}: diferencia relativa = {diferencia:.3f}')

**Observaciones.**

- `CRIM` presenta una diferencia muy grande entre media y mediana: existe una **cola superior pronunciada**.
- `ZN` tiene mediana igual a cero, señal de una fuerte concentración de observaciones en ese valor.
- `RM` presenta media y mediana cercanas, lo que sugiere una distribución mucho más equilibrada que `CRIM`.

## 4. Distribución y valores extremos

**Pregunta orientadora:** ¿Qué valores son habituales y cuáles se alejan claramente del resto?

### 4.1 Histograma textual

`plt.hist` permite estudiar la forma de una distribución sin utilizar librerías gráficas.

In [ ]:
# Visualizar la distribución de algunas variables
variables = ['CRIM', 'RM', 'LSTAT']

for nombre in variables:
    idx = nombres_columnas.index(nombre)
    x = datos[:, idx]

    plt.figure(figsize=(6, 3))
    plt.hist(x, bins=20, edgecolor='black')

    plt.xlabel(nombre)
    plt.ylabel('Frecuencia')
    plt.title(f'Distribución de {nombre}')

    plt.show()

### 4.2 Regla del rango intercuartílico (IQR)

Para variables cuantitativas, un criterio exploratorio común identifica observaciones por debajo de `Q1 - 1.5·IQR` o por encima de `Q3 + 1.5·IQR`.

**Importante:** un valor extremo **no es automáticamente un error** y la regla no debe aplicarse a variables binarias como `CHAS` ni a índices discretos como `RAD`.

In [ ]:
# Variables donde la regla IQR tiene una interpretación cuantitativa razonable
excluir_iqr = {'CHAS', 'RAD'}

print(f'{"Variable":8s} {"IQR":>10s} {"Extremos":>10s}')
print('-' * 32)

for i, nombre in enumerate(nombres_columnas):
    if nombre in excluir_iqr:
        continue
    iqr = q3[i] - q1[i]
    lim_inf = q1[i] - 1.5 * iqr
    lim_sup = q3[i] + 1.5 * iqr
    n_extremos = np.sum((datos[:, i] < lim_inf) | (datos[:, i] > lim_sup))
    print(f'{nombre:8s} {iqr:10.3f} {n_extremos:10d}')

### 4.3 Caso de estudio: `CRIM`

Los siguientes valores son precisamente los que describiría un boxplot, aunque aquí se calculan únicamente con NumPy.

In [ ]:
idx_crim = nombres_columnas.index('CRIM')
crim = datos[:, idx_crim]

crim_q1 = np.percentile(crim, 25)
crim_mediana = np.median(crim)
crim_q3 = np.percentile(crim, 75)
crim_iqr = crim_q3 - crim_q1
lim_inf = crim_q1 - 1.5 * crim_iqr
lim_sup = crim_q3 + 1.5 * crim_iqr
extremos = crim[(crim < lim_inf) | (crim > lim_sup)]

print(f'Media:       {np.mean(crim):.4f}')
print(f'Mediana:     {crim_mediana:.4f}')
print(f'Q1:          {crim_q1:.4f}')
print(f'Q3:          {crim_q3:.4f}')
print(f'IQR:         {crim_iqr:.4f}')
print(f'Límite sup.: {lim_sup:.4f}')
print(f'Extremos:    {extremos.size}')
print(f'Máximo:      {np.max(crim):.4f}')

**Observación.** `CRIM` es fuertemente asimétrica: la mediana está cerca de valores bajos, mientras algunos registros alcanzan valores mucho mayores. Estos casos deben **investigarse**, no eliminarse de forma automática.

### 4.4 Diagrama de bigotes de `CRIM`

**Pregunta orientadora:** ¿Existen observaciones de `CRIM` que se alejan considerablemente del comportamiento típico de los datos?

El diagrama de bigotes permite visualizar la mediana, los cuartiles, la dispersión de los datos y posibles valores atípicos.

In [ ]:
# Seleccionar la variable CRIM
idx_crim = nombres_columnas.index('CRIM')
crim = datos[:, idx_crim]

# Diagrama de bigotes
plt.figure(figsize=(7, 3))

plt.boxplot(crim, orientation='vertical')

plt.xlabel('CRIM')
plt.title('Diagrama de bigotes de CRIM')

plt.show()

**Observación:** La distribución de `CRIM` presenta varios valores alejados del rango central. El diagrama sugiere una fuerte asimetría hacia valores altos y la presencia de posibles valores atípicos.

Estos puntos no deben eliminarse automáticamente: primero es necesario determinar si corresponden a errores o a observaciones válidas con características extremas.

## 5. Análisis de la variable objetivo: `MEDV`

**Pregunta orientadora:** ¿Qué sabemos de la variable que posteriormente queremos explicar o predecir?

### 5.1 Resumen de `MEDV`

In [ ]:
# 5.1 Resumen de MEDV
idx_medv = nombres_columnas.index('MEDV')
medv = datos[:, idx_medv]

# Resumen estadístico
print(f'Mínimo:   {np.min(medv):.2f}')
print(f'Q1:       {np.percentile(medv, 25):.2f}')
print(f'Mediana:  {np.median(medv):.2f}')
print(f'Media:    {np.mean(medv):.2f}')
print(f'Q3:       {np.percentile(medv, 75):.2f}')
print(f'Máximo:   {np.max(medv):.2f}')

# Distribución de MEDV
plt.figure(figsize=(6, 3))
plt.hist(medv, bins=20, edgecolor='black')

plt.xlabel('MEDV')
plt.ylabel('Frecuencia')
plt.title('Distribución de MEDV')

plt.show()

### 5.2 Concentración en el valor máximo

In [ ]:
valor_maximo = np.max(medv)
n_en_maximo = np.sum(medv == valor_maximo)

print('Valor máximo observado:', valor_maximo)
print('Observaciones exactamente en el máximo:', n_en_maximo)

**Observaciones.**

- `MEDV` se concentra alrededor de valores intermedios, pero presenta observaciones altas que merecen revisión.
- En el conjunto estándar aparecen **16 observaciones exactamente en 50**, lo que revela una acumulación artificial en el límite superior y debe tenerse presente al interpretar futuros modelos.
- Consultar por que ocurre esto.

## 6. Relación de las variables con `MEDV`

**Pregunta orientadora:** ¿Qué variables parecen estar más asociadas con el valor de `MEDV`?

### 6.1 Correlación lineal

La correlación de Pearson toma valores entre `-1` y `1`. Resume **asociación lineal**, no causalidad.

In [ ]:
# Matriz de correlación: columnas como variables
corr = np.corrcoef(datos, rowvar=False)
corr_medv = corr[:, idx_medv]

# Ordenar predictores por magnitud de correlación con MEDV
indices = np.arange(len(nombres_columnas) - 1)
orden = indices[np.argsort(np.abs(corr_medv[indices]))[::-1]]

print(f'{"Variable":8s} {"Corr. MEDV":>12s}')
print('-' * 23)
for i in orden:
    print(f'{nombres_columnas[i]:8s} {corr_medv[i]:12.3f}')

**Observaciones.**

- `LSTAT` presenta la asociación lineal negativa más intensa con `MEDV` (aprox. **−0.74**).
- `RM` presenta la asociación positiva más intensa (aprox. **+0.70**).
- `PTRATIO`, `INDUS` y `TAX` muestran asociaciones negativas moderadas.
- Una correlación alta no demuestra causalidad ni garantiza que la relación sea perfectamente lineal.

## 7. Relaciones entre predictores

**Pregunta orientadora:** ¿Hay variables que contienen información muy parecida entre sí?

### 7.1 Correlaciones fuertes entre características

Se muestran únicamente pares con `|r| ≥ 0.70` para evitar una matriz extensa difícil de leer.

In [ ]:
umbral = 0.70
n_predictores = len(nombres_columnas) - 1
pares = []

for i in range(n_predictores):
    for j in range(i + 1, n_predictores):
        r = corr[i, j]
        if np.abs(r) >= umbral:
            pares.append((np.abs(r), r, nombres_columnas[i], nombres_columnas[j]))

# Mayor magnitud primero
pares.sort(reverse=True)

for _, r, a, b in pares:
    print(f'{a:8s} – {b:8s}: {r: .3f}')

**Observaciones.**

- `RAD` y `TAX` presentan una correlación muy alta (aprox. **0.91**).
- También aparecen asociaciones fuertes entre `NOX`–`DIS`, `INDUS`–`NOX`, `AGE`–`DIS` y otras variables urbanas.
- Esto sugiere posible **redundancia de información**; no significa que alguna variable deba eliminarse todavía.

## 8. Conclusiones del EDA inicial

**Pregunta orientadora:** ¿Qué aprendimos antes de construir cualquier modelo?

1. El conjunto estándar contiene **506 observaciones y 14 variables**, sin faltantes ni duplicados.
2. Las variables no son todas del mismo tipo: `CHAS` es binaria y `RAD` es un índice discreto.
3. Algunas distribuciones, especialmente `CRIM` y `ZN`, son fuertemente asimétricas.
4. Los valores extremos deben interpretarse en su contexto y **no eliminarse automáticamente**.
5. `MEDV` presenta una concentración en su valor máximo de 50 que debe considerarse en análisis posteriores.
6. `LSTAT` y `RM` son las variables con mayor asociación lineal, negativa y positiva respectivamente, con `MEDV`.
7. Existen correlaciones fuertes entre algunos predictores, especialmente `RAD` y `TAX`.

### Siguiente paso

El EDA ya permite formular hipótesis sobre los datos. El siguiente paso puede abordar **preparación de variables, separación de datos y construcción de un primer modelo**.